# GW170817 — BlackJAX Nested Sampling with Time+Phase-Marginalized Likelihood

Full Bayesian parameter estimation for GW170817 using:
- **Likelihood**: Relative-binning with analytical tc and φc marginalization
- **Sampler**: BlackJAX acceptance-walk nested sampler (handley-lab fork)
- **Waveform**: mlgw_bns_jax (JAX-native BNS approximant backed by TEOBResumS)
- **Data**: BayesWave-cleaned 1024-s strain, 128-s analysis segment, 4096 Hz

By marginalizing over tc (3000-point grid, ±150 ms) and φc analytically,
the sampler operates in a 9-dimensional parameter space, avoiding the sharp
tc peak that hinders convergence when tc is explicitly sampled.

Sampled parameters: ln d_L, θ_JN, ψ, M_c, q, χ₁, χ₂, Λ₁, Λ₂
Fixed: RA/Dec = NGC 4993, tc and φc marginalized.

In [ ]:
# ── Google Colab environment setup ──────────────────────────────────────
# Run this cell first on Colab; it is a no-op on local machines.
import os, subprocess, sys

COLAB    = "google.colab" in sys.modules
REPO_DIR = "/content/mlgw_bns_jax" if COLAB else os.getcwd()

if COLAB:
    # ── JAX with CUDA 12 ──────────────────────────────────────────────
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "jax[cuda12]",
        "-f", "https://storage.googleapis.com/jax-releases/jax_cuda_releases.html",
    ])
    # ── Other Python dependencies ──────────────────────────────────────
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "gwpy", "h5py", "corner", "tqdm", "anesthetic",
    ])
    # ── BlackJAX (nested_sampling branch, pinned commit) ──────────────
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "blackjax@git+https://github.com/handley-lab/blackjax.git"
        "@dedbf11da33eb5ca286f6731e2c51f2b254b953f",
    ])

    # ── Clone the mlgw_bns_jax repo (model + waveform loader) ─────────
    if not os.path.isdir(REPO_DIR):
        subprocess.check_call([
            "git", "clone", "--branch", "time-marg-likelihood-pe",
            "--depth", "1",
            "https://github.com/saulo-albuquerque-phys/mlgw_bns_jax.git",
            REPO_DIR,
        ])

    # ── Clone blackjax_ns_gw (custom NS kernels) ──────────────────────
    _ns_repo   = os.path.join(REPO_DIR, "_blackjax_ns_gw_repo")
    _ns_src    = os.path.join(_ns_repo,  "src", "custom_kernels")
    _ns_link   = os.path.join(REPO_DIR,  "custom_kernels")
    if not os.path.isdir(_ns_repo):
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/mrosep/blackjax_ns_gw.git",
            _ns_repo,
        ])
    if not os.path.exists(_ns_link):
        os.symlink(_ns_src, _ns_link)

    # ── Clone SHARPy (GWNetwork / data loading) ───────────────────────
    _sharpy_repo = os.path.join(REPO_DIR, "_sharpy_repo")
    _sharpy_pkg  = os.path.join(_sharpy_repo, "sharpy")
    _sharpy_link = os.path.join(REPO_DIR, "sharpy")
    if not os.path.isdir(_sharpy_repo):
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/saulo-albuquerque-phys/sharpy.git",
            _sharpy_repo,
        ])
    if not os.path.exists(_sharpy_link):
        os.symlink(_sharpy_pkg, _sharpy_link)

    os.chdir(REPO_DIR)
    print(f"Working directory  : {os.getcwd()}")
    print(f"BlackJAX-NS kernels: {_ns_link}")
    print(f"SHARPy symlink     : {_sharpy_link}")
else:
    print("Not running on Colab — skipping environment setup.")


In [ ]:
from __future__ import annotations
import os, sys, time, pickle
import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

print("JAX devices:", jax.devices())

# ── Event & data configuration ───────────────────────────────────────────
TRIGGER_TIME     = 1187008882.43
SEGMENT_DURATION = 128.0
SAMPLING_RATE    = 4096
F_LOWER          = 23.0
F_UPPER          = 2000.0
DATA_START_GPS   = 1187008114
DATA_DURATION    = 1024

FIXED_RA  = 3.44616      # rad (NGC 4993)
FIXED_DEC = -0.408084    # rad

DATA_DIR = "gw170817_data"
OUTDIR   = "outdir_time_marg_pe"
LABEL    = "GW170817_time_marg_pe"
os.makedirs(OUTDIR, exist_ok=True)

# ── tc grid: 3000 points in [-0.15, 0.15] s ─────────────────────────────
N_TC    = 3000
TC_MIN  = -0.15
TC_MAX  =  0.15
TC_GRID = np.linspace(TC_MIN, TC_MAX, N_TC)

# ── RB settings ──────────────────────────────────────────────────────────
N_BINS  = 400

# ── Sampler settings ──────────────────────────────────────────────────────
N_LIVE   = 1400
N_DELETE = N_LIVE // 2
N_TARGET = 60
MAX_MCMC = 5000
SEED     = 42
DLOGZ_STOP = 0.1

print(f"Segment: {SEGMENT_DURATION} s @ {SAMPLING_RATE} Hz")
print(f"tc grid: {N_TC} pts in [{TC_MIN}, {TC_MAX}] s")
print(f"RB bins: {N_BINS}")
print(f"Sampler: {N_LIVE} live points, batch {N_DELETE}")

In [ ]:
from jax_import_n_predict import load_predict

MODEL_PATH = "mlgw_bns_jax_model.h5"
assert os.path.exists(MODEL_PATH), f"Model not found: {MODEL_PATH}"
_mlgw_predict = load_predict(MODEL_PATH)

import sharpy.GW_likelihood as _gw_mod
from sharpy.utils import McQ2Masses

def _template_mlgw_bns(params, frequency_array):
    """JAX-jittable mlgw_bns_jax waveform template."""
    mc, q    = params[6], params[7]
    m1, m2   = McQ2Masses(mc, q)
    total_mass = m1 + m2
    chi1, chi2    = params[9], params[10]
    lambda_1, lambda_2 = params[11], params[12]
    phic      = params[4]
    dist_mpc  = jnp.exp(params[2])
    inclination = params[3]
    mlgw_params = jnp.array([q, lambda_1, lambda_2, chi1, chi2])
    hp, hc = _mlgw_predict(
        mlgw_params, frequency_array,
        total_mass=total_mass, distance_mpc=dist_mpc, inclination=inclination,
    )
    phase_factor = jnp.exp(-1j * phic)
    return hp * phase_factor, hc * phase_factor

_gw_mod.template = _template_mlgw_bns

from sharpy.GW_likelihood import GWNetwork, log_likelihood_det
import sharpy.PSDs
print("mlgw_bns_jax loaded and SHARPy template patched.")

In [ ]:
data_files = {
    "H1": os.path.join(DATA_DIR, f"H-H1_BWCLEANED_4KHZ-{DATA_START_GPS}-{DATA_DURATION}.txt"),
    "L1": os.path.join(DATA_DIR, f"L-L1_BWCLEANED_4KHZ-{DATA_START_GPS}-{DATA_DURATION}.txt"),
    "V1": os.path.join(DATA_DIR, f"V-V1_BWCLEANED_4KHZ-{DATA_START_GPS}-{DATA_DURATION}.txt"),
}
for det, f in data_files.items():
    assert os.path.isfile(f), f"Missing data: {f}"
    print(f"{det}: {os.path.basename(f)}")

detector_settings = {}
for det in ["H1", "L1", "V1"]:
    detector_settings[det] = dict(
        data_file=data_files[det], channel="GWOSC",
        trigger_time=TRIGGER_TIME, duration=SEGMENT_DURATION,
        sampling_rate=SAMPLING_RATE, f_lower=F_LOWER, f_upper=F_UPPER,
        psd_file=None, psd_method="welch",
        download_data=False, zero_noise=False,
    )

print(f"\nBuilding GW network ({SEGMENT_DURATION}s segment)...")
t0 = time.time()
gw_network = GWNetwork(detector_settings, injection_parameters=None)
print(f"Network built in {time.time()-t0:.2f} s")
batched_detector = gw_network.batched_detector

## Q-transform spectrograms — BayesWave-cleaned data

We verify that we are using the correct BayesWave-deglitched data by plotting
the Q-transform spectrograms for all three detectors.  The scattered-light
glitch in L1 that was present before GW170817 is removed in the cleaned data.

In [ ]:
from gwpy.timeseries import TimeSeries

MERGER_GPS   = TRIGGER_TIME
T_START_PLOT = MERGER_GPS - 3.0
T_END_PLOT   = MERGER_GPS + 3.0
F_PLOT_MIN, F_PLOT_MAX = 20.0, 800.0

DET_COLORS = {"H1": "Reds", "L1": "Blues", "V1": "Purples"}
DET_LABELS = {"H1": "LIGO Hanford (H1)", "L1": "LIGO Livingston (L1)", "V1": "Virgo (V1)"}

def _qtransform(filepath):
    strain = np.loadtxt(filepath, comments="#")
    ts = TimeSeries(strain, sample_rate=SAMPLING_RATE, t0=DATA_START_GPS)
    ts_w = ts.whiten(4, 2)
    ts_c = ts_w.crop(T_START_PLOT - 1, T_END_PLOT + 1)
    return ts_c.q_transform(frange=(F_PLOT_MIN, F_PLOT_MAX), qrange=(4, 64),
                             outseg=(T_START_PLOT, T_END_PLOT), logf=True)

# ── Raw vs BayesWave-cleaned L1 comparison ──────────────────────────────
raw_l1 = os.path.join(DATA_DIR, f"L-L1_GWOSC_4KHZ_R1-{DATA_START_GPS}-{DATA_DURATION}.txt")
if os.path.isfile(raw_l1):
    qt_raw = _qtransform(raw_l1)
    qt_cln = _qtransform(data_files["L1"])
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 5), sharey=True)
    for ax, qt, title in [(ax1, qt_raw, "L1 — Raw GWOSC (with glitch)"),
                           (ax2, qt_cln, "L1 — BayesWave Cleaned")]:
        pcm = ax.pcolormesh(qt.times.value - MERGER_GPS, qt.frequencies.value,
                             qt.value.T, cmap="Blues", vmin=0, vmax=25)
        ax.set_yscale("log"); ax.set_ylim(F_PLOT_MIN, F_PLOT_MAX)
        ax.set_xlabel("Time relative to merger [s]", fontsize=13)
        ax.set_title(title, fontsize=14)
        ax.axvline(0, color="white", ls="--", lw=1.5, alpha=0.8, label="Merger")
        ax.legend(fontsize=11)
        fig.colorbar(pcm, ax=ax).set_label("Normalized energy")
    ax1.set_ylabel("Frequency [Hz]", fontsize=13)
    fig.suptitle("GW170817 — L1 glitch comparison", fontsize=15)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTDIR, f"{LABEL}_L1_comparison.png"), dpi=150)
    plt.show()

# ── All three detectors ──────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 14), sharex=True)
for ax, det in zip(axes, ["H1", "L1", "V1"]):
    qt = _qtransform(data_files[det])
    pcm = ax.pcolormesh(qt.times.value - MERGER_GPS, qt.frequencies.value,
                         qt.value.T, cmap=DET_COLORS[det], vmin=0, vmax=25)
    ax.set_yscale("log"); ax.set_ylim(F_PLOT_MIN, F_PLOT_MAX)
    ax.set_ylabel("Frequency [Hz]", fontsize=13)
    ax.set_title(f"{DET_LABELS[det]} — BayesWave cleaned", fontsize=13)
    ax.axvline(0, color="white", ls="--", lw=1.5, alpha=0.8)
    fig.colorbar(pcm, ax=ax).set_label("Normalized energy")
axes[-1].set_xlabel("Time relative to merger [s]", fontsize=13)
fig.suptitle("GW170817 — Q-transform spectrograms (BayesWave cleaned, 1024 s data)",
             fontsize=15, y=0.995)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, f"{LABEL}_qtransform_bwcleaned.png"), dpi=150)
plt.show()
print("Spectrograms saved.")

## Build the time+phase-marginalized RB likelihood

We use the `build_rb_likelihood_tc_phi_marg` function which:
1. Builds the standard RB summary data (A0, B0, dd) with 400 bins
2. Pre-computes phase-stripped A0 matrices for the φc marginalization
3. Returns a function that, for each candidate (9-param) point:
   - Computes r_j⁰ at bin centres (at fiducial tc, phic=0)
   - Computes W(tc_k) = Σ_det TwoDTN · r⁰ · A0_nophase · exp(-i2πf Δtc)  via matrix multiply
   - Returns: const + logsumexp_k [log I₀(2|W(tc_k)|)] - log N_tc

In [ ]:
from relative_binning import build_rb_likelihood_tc_phi_marg

# ── Fiducial parameters (approximate MAP from arXiv:2210.15684) ──────────
# [ra, dec, logdist, incl, phic, pol, mc, q, tc, chi1, chi2, lambda1, lambda2]
FIDUCIAL_PARAMS = np.array([
    FIXED_RA,          # [0]  ra  (NGC 4993)
    FIXED_DEC,         # [1]  dec
    np.log(40.0),      # [2]  logdist — 40 Mpc
    2.545,             # [3]  theta_jn — ~146 degrees (edge-on favoured)
    0.0,               # [4]  phic (will be marginalized)
    0.0,               # [5]  pol
    1.1975,            # [6]  mc  [M_sun]
    0.87,              # [7]  q
    0.0,               # [8]  tc  (will be marginalized, relative to trigger)
    0.0,               # [9]  chi1
    0.0,               # [10] chi2
    400.0,             # [11] lambda1
    400.0,             # [12] lambda2
])

print("Building time+phase-marginalized RB likelihood...")
t0 = time.time()
log_L_tp_marg_rb, rb_network = build_rb_likelihood_tc_phi_marg(
    batched_detector, FIDUCIAL_PARAMS, _template_mlgw_bns,
    TC_GRID, n_bins=N_BINS,
)
print(f"Built in {time.time()-t0:.2f} s")

# ── JIT compile & warm up ────────────────────────────────────────────────
log_L_tp_marg_rb_jit = jax.jit(log_L_tp_marg_rb)

# 11-param array: [ra, dec, logdist, incl, pol, mc, q, chi1, chi2, lambda1, lambda2]
# (tc at index 8 removed, phic at index 4 removed → pol shifts to index 4)
_p11_fid = FIDUCIAL_PARAMS[[0,1,2,3,5,6,7,9,10,11,12]]

t_warmup = time.time()
_ = float(log_L_tp_marg_rb_jit(jnp.array(_p11_fid)))
print(f"JIT warm-up: {time.time()-t_warmup:.2f} s")
print(f"log L at fiducial (tc+φ marg, RB): {float(log_L_tp_marg_rb_jit(jnp.array(_p11_fid))):.2f}")

n_full  = len(np.array(batched_detector.Frequency[0]))
n_bins  = len(np.array(rb_network.f_bins))
print(f"\nFrequency grid: {n_full} pts → RB bins: {n_bins} pts ({n_full//n_bins}× reduction)")
print(f"tc grid: {N_TC} pts  →  total log L evaluations per call: {N_TC} × {n_bins} matrix multiply")

## Configure BlackJAX nested sampler

We use the acceptance-walk nested sampler from
[blackjax_ns_gw](https://github.com/mrosep/blackjax_ns_gw).

**9 sampled parameters** (tc and φc analytically marginalized):

| Parameter | Prior | Notes |
|-----------|-------|-------|
| ln d_L    | [ln 1, ln 75] Mpc | Flat in log distance |
| θ_JN      | [0, π] | Inclination angle |
| ψ         | [0, π] | Polarization angle |
| M_c       | [1.18, 1.21] M☉ | Chirp mass |
| q         | [0.5, 1.0] | Mass ratio |
| χ₁        | [−0.5, 0.5] | Aligned spin |
| χ₂        | [−0.5, 0.5] | Aligned spin |
| Λ₁        | [5, 5000] | Tidal deformability |
| Λ₂        | [5, 5000] | Tidal deformability |

In [ ]:
import tqdm

from custom_kernels import (
    acceptance_walk_sampler,
    create_unit_cube_functions,
    init_unit_cube_particles,
    transform_to_physical,
)

# ── Parameter names (9, with tc and phic analytically marginalized) ──────
# Note: params_no_tc_no_phi order: [ra, dec, logdist, incl, pol, mc, q, chi1, chi2, λ1, λ2]
# We sample only the non-fixed, non-marginalized subset:
parameter_names = [
    "logdistance", "theta_jn", "pol",
    "mc", "q", "chi1", "chi2", "lambda_1", "lambda_2",
]

# ── Prior bounds ──────────────────────────────────────────────────────────
param_bounds = {
    "logdistance": (jnp.log(1.0),   jnp.log(75.0)),
    "theta_jn":    (0.0,             jnp.pi),
    "pol":         (0.0,             jnp.pi),
    "mc":          (1.18,            1.21),
    "q":           (0.5,             1.0),
    "chi1":        (-0.5,            0.5),
    "chi2":        (-0.5,            0.5),
    "lambda_1":    (5.0,             5000.0),
    "lambda_2":    (5.0,             5000.0),
}

# ── Build 11-param array for the likelihood from the 9 sampled params ────
def loglikelihood_from_dict(params_dict):
    """Map 9-param BlackJAX-NS dict → 11-param array → tc+φ-marg RB log L."""
    # Full 11-param array: [ra, dec, logdist, incl, pol, mc, q, chi1, chi2, λ1, λ2]
    params_11 = jnp.array([
        FIXED_RA,                        # [0] ra  (fixed)
        FIXED_DEC,                       # [1] dec (fixed)
        params_dict["logdistance"],      # [2]
        params_dict["theta_jn"],         # [3]
        params_dict["pol"],              # [4]
        params_dict["mc"],               # [5]
        params_dict["q"],                # [6]
        params_dict["chi1"],             # [7]
        params_dict["chi2"],             # [8]
        params_dict["lambda_1"],         # [9]
        params_dict["lambda_2"],         # [10]
    ])
    return log_L_tp_marg_rb_jit(params_11)

# Determine JAX's ravel order for the parameter dict
_example = {key: float(i) for i, key in enumerate(parameter_names)}
_flat, _ = jax.flatten_util.ravel_pytree(_example)
_ravel_order = []
for val in _flat:
    for key, test_val in _example.items():
        if abs(val - test_val) < 1e-10:
            _ravel_order.append(key)
            break
print(f"JAX ravel order: {_ravel_order}")

param_mins = jnp.array([param_bounds[k][0] for k in _ravel_order])
param_maxs = jnp.array([param_bounds[k][1] for k in _ravel_order])
n_dims     = len(parameter_names)

# ── Unit-cube prior transform ─────────────────────────────────────────────
@jax.jit
def prior_transform_fn(u_params):
    u_values, _ = jax.flatten_util.ravel_pytree(u_params)
    x_values = param_mins + u_values * (param_maxs - param_mins)
    example = {key: 0.0 for key in parameter_names}
    _, unflatten_fn = jax.flatten_util.ravel_pytree(example)
    return unflatten_fn(x_values)

@jax.jit
def logprior_fn(params):
    param_values, _ = jax.flatten_util.ravel_pytree(params)
    in_bounds = jnp.all((param_values >= param_mins) & (param_values <= param_maxs))
    log_vol   = jnp.sum(jnp.log(param_maxs - param_mins))
    return jnp.where(in_bounds, -log_vol, -jnp.inf)

print(f"Sampling {n_dims} parameters: {parameter_names}")
print(f"Analytically marginalized: tc (over {N_TC}-pt grid), φc (Bessel function)")

In [ ]:
rng_key = jax.random.PRNGKey(SEED)
rng_key, init_key = jax.random.split(rng_key)

example_params = {key: 0.0 for key in parameter_names}

# ── Initialize live points ───────────────────────────────────────────────
unit_cube_particles = init_unit_cube_particles(init_key, example_params, N_LIVE)

# ── Periodic parameter mask (pol is periodic in [0, π]) ─────────────────
periodic_mask = jax.tree_util.tree_map(lambda _: False, example_params)
periodic_mask["pol"] = True

# ── Create unit-cube wrappers ────────────────────────────────────────────
unit_cube_fns = create_unit_cube_functions(
    physical_loglikelihood_fn=loglikelihood_from_dict,
    prior_transform_fn=prior_transform_fn,
    mask_tree=periodic_mask,
)

# ── Build nested sampler ─────────────────────────────────────────────────
nested_sampler = acceptance_walk_sampler(
    logprior_fn=unit_cube_fns["logprior_fn"],
    loglikelihood_fn=unit_cube_fns["loglikelihood_fn"],
    nlive=N_LIVE,
    n_target=N_TARGET,
    max_mcmc=MAX_MCMC,
    num_delete=N_DELETE,
    stepper_fn=unit_cube_fns["stepper_fn"],
)

state = nested_sampler.init(unit_cube_particles)

print(f"BlackJAX-NS configured:")
print(f"  Live points:     {N_LIVE}")
print(f"  Batch delete:    {N_DELETE}")
print(f"  Target walks:    {N_TARGET}")
print(f"  Max MCMC:        {MAX_MCMC}")
print(f"  Stop at ΔlnZ <   {DLOGZ_STOP}")
print(f"  Seed:            {SEED}")

## Run nested sampling

The loop runs until the estimated remaining log-evidence
Δln Z = ln(Z_live / Z) < 0.1.  Each iteration removes `N_DELETE`
dead points and replaces them via the acceptance-walk kernel.

A checkpoint is saved every 200 iterations to `outdir_time_marg_pe/`.

In [ ]:
@jax.jit
def one_step(carry, xs):
    state, k = carry
    k, subk = jax.random.split(k)
    state, dead_point = nested_sampler.step(subk, state)
    return (state, k), dead_point


def terminate(state):
    dlogz = jnp.logaddexp(0, state.logZ_live - state.logZ)
    return bool(jnp.isfinite(dlogz) and dlogz < DLOGZ_STOP)


print(f"Starting BlackJAX-NS with {N_LIVE} live points (tc+φ-marg, 9-D space)...")
t_start = time.time()
dead    = []
n_iter  = 0

CHECKPOINT_EVERY = 200  # save state every N iterations

with tqdm.tqdm(desc="Dead pts", unit=" pts") as pbar:
    while not terminate(state):
        (state, rng_key), dead_info = one_step((state, rng_key), None)
        dead.append(dead_info)
        n_iter += 1
        pbar.update(N_DELETE)

        # Periodic checkpoint
        if n_iter % CHECKPOINT_EVERY == 0:
            ckpt = {"state": state, "dead": dead, "n_iter": n_iter}
            with open(os.path.join(OUTDIR, f"{LABEL}_checkpoint.pkl"), "wb") as f:
                pickle.dump(ckpt, f)

dt = time.time() - t_start
n_dead_total = len(dead) * N_DELETE
print(f"\nDone in {dt:.1f} s — {n_dead_total} dead points")
print(f"log Z (running):  {state.logZ:.3f}")
print(f"log Z (live):     {state.logZ_live:.3f}")
print(f"Δlog Z:           {float(jnp.logaddexp(0, state.logZ_live - state.logZ)):.4f}")

In [ ]:
from blackjax.ns.utils import finalise
from anesthetic import NestedSamples

# ── Finalise: merge dead + live ──────────────────────────────────────────
final_state = finalise(state, dead)

# ── Save ─────────────────────────────────────────────────────────────────
state_path = os.path.join(OUTDIR, f"{LABEL}_final_state.pkl")
with open(state_path, "wb") as f:
    pickle.dump(final_state, f)
print(f"Final state saved: {state_path}")

# ── Transform unit-cube → physical space ─────────────────────────────────
physical_particles = transform_to_physical(final_state.particles, prior_transform_fn)

# ── Column labels for anesthetic ─────────────────────────────────────────
column_to_label = {
    "logdistance": r"$\ln d_L$",
    "theta_jn":    r"$\theta_{JN}$",
    "pol":         r"$\psi$",
    "mc":          r"$\mathcal{M}_c$",
    "q":           r"$q$",
    "chi1":        r"$\chi_1$",
    "chi2":        r"$\chi_2$",
    "lambda_1":    r"$\Lambda_1$",
    "lambda_2":    r"$\Lambda_2$",
}

logL_birth = final_state.loglikelihood_birth.copy()
logL_birth = jnp.where(jnp.isnan(logL_birth), -jnp.inf, logL_birth)

samples_anesthetic = NestedSamples(
    physical_particles,
    logL=final_state.loglikelihood,
    logL_birth=logL_birth,
    labels=column_to_label,
    logzero=jnp.nan,
    dtype=jnp.float64,
)

csv_path = os.path.join(OUTDIR, f"{LABEL}_posterior.csv")
samples_anesthetic.to_csv(csv_path)
print(f"Posterior samples saved: {csv_path}")

logZ_samples = samples_anesthetic.logZ(100)
print(f"\nBayesian evidence:")
print(f"  ln Z (integrator): {state.logZ:.3f}")
print(f"  ln Z (anesthetic): {logZ_samples.mean():.3f} ± {logZ_samples.std():.3f}")
print(f"  Dead points:       {n_dead_total}")

In [ ]:
from corner import corner as corner_plot

# ── Build equally-weighted sample array ──────────────────────────────────
eq_samples  = samples_anesthetic.sample(n=5000, replace=True)
eq_array    = np.column_stack([np.array(eq_samples[k]) for k in _ravel_order])
ravel_to_param_idx = [_ravel_order.index(k) for k in parameter_names]
samples = eq_array[:, ravel_to_param_idx]

print(f"Equally-weighted samples: {samples.shape}")
print(f"Columns: {parameter_names}")

# ── Full corner plot ──────────────────────────────────────────────────────
fig = corner_plot(samples, show_titles=True,
                  labels=parameter_names, title_kwargs={"fontsize": 9})
fig.suptitle("GW170817 — BlackJAX-NS (tc+φ marginalised, 9-D)", fontsize=14, y=1.01)
corner_path = os.path.join(OUTDIR, f"{LABEL}_corner_full.png")
fig.savefig(corner_path, dpi=150, bbox_inches="tight")
print(f"Corner plot saved: {corner_path}")
plt.show()

## Paper-style results (Figure 9 of arXiv:2210.15684)

Compute derived parameters and compare with the paper's MAP values:
- M_c = 1.1975 M☉
- q ≈ 0.87
- χ_eff ≈ 0.0
- Λ̃ ≈ 300–400
- D_L ≈ 40 Mpc

Column map for the 9-dim samples array:
`[0] logdist, [1] theta_jn, [2] pol, [3] mc, [4] q, [5] chi1, [6] chi2, [7] lambda_1, [8] lambda_2`

In [ ]:
# ── Derived parameters ────────────────────────────────────────────────────
# samples column order from parameter_names:
# 0:logdist, 1:theta_jn, 2:pol, 3:mc, 4:q, 5:chi1, 6:chi2, 7:lambda_1, 8:lambda_2

mc_s     = samples[:, 3]
q_s      = samples[:, 4]
chi1_s   = samples[:, 5]
chi2_s   = samples[:, 6]
lam1_s   = samples[:, 7]
lam2_s   = samples[:, 8]
logdist_s = samples[:, 0]

chi_eff = (chi1_s + q_s * chi2_s) / (1.0 + q_s)
lambda_tilde = (16.0 / 13.0) * (
    (12.0 * q_s + 1.0) * lam1_s + (12.0 + q_s) * q_s**4 * lam2_s
) / (1.0 + q_s)**5
d_L_mpc = np.exp(logdist_s)

paper_samples = np.column_stack([mc_s, q_s, chi_eff, lambda_tilde, d_L_mpc])
paper_labels  = [
    r"$\mathcal{M}_c\;[M_\odot]$",
    r"$q$",
    r"$\chi_\mathrm{eff}$",
    r"$\tilde{\Lambda}$",
    r"$D_L\;[\mathrm{Mpc}]$",
]

# Paper MAP truths (arXiv:2210.15684, Table I)
truths = [1.1975, 0.87, 0.0, 330.0, 40.0]

fig = corner_plot(
    paper_samples, labels=paper_labels,
    show_titles=True, title_kwargs={"fontsize": 10},
    truths=truths, truth_color="tab:red",
)
fig.suptitle("GW170817 — BlackJAX-NS tc+φ-marg (9-D)\nRed: paper MAP values",
             fontsize=13, y=1.01)
paper_path = os.path.join(OUTDIR, f"{LABEL}_corner_paper.png")
fig.savefig(paper_path, dpi=150, bbox_inches="tight")
print(f"Paper-style corner saved: {paper_path}")
plt.show()

# ── Print median and 90% CI ──────────────────────────────────────────────
print("\nMedian ± 90% CI (5th–95th percentiles):")
for label, col_data in zip(["M_c", "q", "chi_eff", "Lambda_tilde", "D_L"],
                             [mc_s, q_s, chi_eff, lambda_tilde, d_L_mpc]):
    lo, med, hi = np.percentile(col_data, [5, 50, 95])
    print(f"  {label:15s}: {med:.3f}  [{lo:.3f}, {hi:.3f}]")

In [ ]:
print("=" * 70)
print("SUMMARY — GW170817 BlackJAX-NS (tc+φ marginalized)")
print("=" * 70)
print(f"\nData:     BayesWave-cleaned, 128 s, {SAMPLING_RATE} Hz")
print(f"Waveform: mlgw_bns_jax")
print(f"tc grid:  {N_TC} pts in [{TC_MIN}, {TC_MAX}] s")
print(f"RB bins:  {len(np.array(rb_network.f_bins))} (of {len(np.array(batched_detector.Frequency[0]))} full-grid pts)")
print(f"\nSampler:  BlackJAX acceptance-walk nested sampling")
print(f"Dims:     {n_dims} (tc + φc analytically marginalized)")
print(f"Live pts: {N_LIVE}, batch: {N_DELETE}")
print(f"\nResults:")
print(f"  ln Z = {state.logZ:.3f}")
print(f"  Dead pts: {n_dead_total}")
print(f"  Wall time: {dt:.1f} s ({dt/3600:.2f} h)")
print(f"\nPosterior medians (50th percentile):")
for lbl, arr in [("M_c [M_sun]", mc_s), ("q", q_s), ("chi_eff", chi_eff),
                  ("Lambda_tilde", lambda_tilde), ("D_L [Mpc]", d_L_mpc)]:
    print(f"  {lbl:20s}: {np.median(arr):.3f}")